# scGPT Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 11.08.2025
- Date of Last Modification: 12.08.2025 (Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the scgpt conda environment, installable from ```../../envs/scgpt.yaml```.
- Clone the Github repository ```https://github.com/bowang-lab/scGPT.git``` into ```<root>/analysis/benchmarking```.
- Load the model checkpoint from ```https://drive.google.com/drive/folders/1_GROJTzXiAV8HB4imruOTk6PEGuNOcgB``` as described at ```https://github.com/bowang-lab/scGPT``` and store in ```<root>/analysis/benchmarking/scGPT/checkpoints```.
- Download the metadata from ```https://github.com/bowang-lab/scGPT?tab=readme-ov-file#pretrained-scGPT-checkpoints``` and store in ```<root>/analysis/benchmarking/scGPT/metadata```.

# 1. Setup

## 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import pickle
from pathlib import Path

import anndata as ad
import numpy as np
import omnipath as op
import pandas as pd
import scanpy as sc
import scgpt
import scipy.spatial.distance as dist
import torch
import torch.nn as nn
import torch.nn.functional as F
from cellphonedb.utils import db_releases_utils, db_utils
from IPython.display import HTML, display
from pyensembl import EnsemblRelease
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.metrics import (classification_report,
                             mean_absolute_error,
                             mean_squared_error)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, random_split

/nfs/team361/sb75/.venvs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2 Configure Paths

In [3]:
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'
scgpt_model_dir = Path("./scGPT/checkpoints")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("./scGPT/metadata/gene_info.csv")

## 1.3 Define Functions

In [4]:
# from /terra/evaluation/linear.py
class LinearClassifier(nn.Module):
    """Linear layer to train on top of frozen features"""
    def __init__(self, num_features: int, num_classes: int):
        super(LinearClassifier, self).__init__()
        self.num_classes = num_classes
        self.linear = nn.Linear(num_features, num_classes)
        self.linear.weight.data.normal_(mean=0.0, std=0.01)
        self.linear.bias.data.zero_()

    def forward(self, x):
        # Flatten
        x = x.view(x.size(0), -1)

        return self.linear(x)


# from /terra/evaluation/linear.py
def linear_classifier(
    features_train: np.ndarray,
    labels_train: np.ndarray,
    features_val: np.ndarray,
    labels_val: np.ndarray,
    features_test: np.ndarray,
    labels_test: np.ndarray,
    n_epochs: int = 400,
    batch_size: int = 128,
    lr: float = 0.001,
    patience: int = 10,
    results_save_path: str | None = None,
    ):
    """
    Train a linear classifier with early stopping on validation loss.
    """

    # --- Device setup ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    features_train = torch.tensor(features_train, dtype=torch.float32)
    labels_train = torch.tensor(labels_train, dtype=torch.long)
    features_val = torch.tensor(features_val, dtype=torch.float32)
    labels_val = torch.tensor(labels_val, dtype=torch.long)
    features_test = torch.tensor(features_test, dtype=torch.float32)
    labels_test = torch.tensor(labels_test, dtype=torch.long)

    train_dataset = TensorDataset(features_train, labels_train)
    val_dataset = TensorDataset(features_val, labels_val)
    test_dataset = TensorDataset(features_test, labels_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # --- Model, Optimizer, Scheduler ---
    num_features = features_train.shape[1]
    num_classes = len(torch.unique(labels_train)) # all classes need to be in train split
    model = LinearClassifier(num_features, num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr * batch_size / 256., # linear scaling rule
        momentum=0.9,
        weight_decay=0,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=0)

    # --- Early Stopping ---
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    # --- Training Loop ---
    for epoch in range(n_epochs):
        model.train()
        train_loss = 0
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- Validation Loop ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features = batch_features.to(device)
                batch_labels = batch_labels.to(device)

                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # --- Load best model before test ---
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # --- Test Evaluation ---
    model.eval()
    all_logits = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            
            _, predicted = torch.max(outputs, 1)

            all_logits.extend(outputs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())

    print("\n--- Evaluation Report on Test Set ---")
    cls_report = classification_report(
        all_targets, all_preds, digits=4)
    print(cls_report)

    # Save to a .txt file
    if results_save_path:
        with open(results_save_path, "w") as f:
            f.write(cls_report)

        print("\n--- Evaluation Report saved. ---")

    return all_preds, all_targets, all_logits, model


# from /terra/evaluation/lr_interactions.py
def get_adata_cellphonedb_lr_pairs(
        adata: ad.AnnData,
        save_folder_path: str,
        cpdb_version: str = 'v5.0.0',
        ) -> pd.DataFrame:
    """
    Retrieve ligand-receptor pairs from CellPhoneDB
    (https://www.cellphonedb.org/) that are present in an AnnData object.

    Parameters
    -----------
    adata:
        The AnnData object to check for gene availability
    save_folder_path

    Returns
    -----------
    lr_interaction_df:
        A DataFrame containing ligand and receptor pairs present in the
        AnnData object.
    """
    # Display CellPhoneDB version
    display(HTML(db_releases_utils.get_remote_database_versions_html()[
        'db_releases_html_table']))

    # Path where the input files to generate the database are/will be located
    os.makedirs(save_folder_path, exist_ok=True)
    cpdb_target_dir = os.path.join(save_folder_path, cpdb_version)

    # Download the CellPhoneDB database
    db_utils.download_database(cpdb_target_dir, cpdb_version)

    # Retrieve CellPhoneDB interactions
    intercell_df = pd.read_csv(f'{cpdb_target_dir}/interaction_input.csv')

    # Filter for lr interactions
    lr_interaction_df = intercell_df[
        intercell_df['directionality'] == 'Ligand-Receptor']

    # Remove interactions involving protein complexes
    lr_interaction_df = lr_interaction_df[
        ~lr_interaction_df['interactors'].apply(lambda x: '+' in x)]

    # Only keep ligand and receptor gene names
    lr_interaction_df['ligand'] = lr_interaction_df[
        'interactors'].apply(lambda x: x.split('-')[0])
    lr_interaction_df['receptor'] = lr_interaction_df[
        'interactors'].apply(lambda x: x.split('-')[1])
    lr_interaction_df = lr_interaction_df[['ligand', 'receptor']]

    # Map gene names to ensembl IDs, keeping only genes present in the AnnData
    # object
    lr_interaction_df['ligand'] = lr_interaction_df[
        'ligand'].map(adata.var['ensembl_id'])
    lr_interaction_df['receptor'] = lr_interaction_df[
        'receptor'].map(adata.var['ensembl_id'])

    print()

    # Drop interactions where ligand or receptor is not in the AnnData object
    lr_interaction_df = lr_interaction_df.dropna()

    return lr_interaction_df


# from /terra/evaluation/lr_interactions.py
def get_adata_omnipath_lr_pairs(
        adata: ad.AnnData,
        min_curation_effort: int = 2
        ) -> pd.DataFrame:
    """
    Retrieve ligand-receptor pairs from OmniPath (https://omnipathdb.org/) that
    are present in an AnnData object.

    Parameters
    -----------
    adata:
        The AnnData object to check for gene availability
    min_curation_effort:
        Minimum curation effort parameter of the OmniPath database.

    Returns
    -----------
    lr_interaction_df:
        A DataFrame containing ligand and receptor pairs present in the
        AnnData object.
    """
    # Retrieve OmniPath interactions
    intercell_df = op.interactions.import_intercell_network(
        include=['omnipath', 'pathwayextra', 'ligrecextra'])

    # Filter for lr interactions
    lr_interaction_df = intercell_df[
        (intercell_df['category_intercell_source'] == 'ligand') &
        (intercell_df['category_intercell_target'] == 'receptor')]

    # Filter for minimum curation effort
    lr_interaction_df = lr_interaction_df[
        lr_interaction_df['curation_effort'] >= min_curation_effort]

    # Remove interactions involving protein complexes
    lr_interaction_df = lr_interaction_df[
        ~lr_interaction_df['genesymbol_intercell_source'].apply(
            lambda x: 'COMPLEX' in x)]
    lr_interaction_df = lr_interaction_df[
        ~lr_interaction_df['genesymbol_intercell_target'].apply(
            lambda x: 'COMPLEX' in x)]

    # Only keep ligand and receptor gene names
    lr_interaction_df = lr_interaction_df[
        ['genesymbol_intercell_source', 'genesymbol_intercell_target']]

    # Map gene names to ensembl IDs, keeping only genes present in the AnnData
    # object
    lr_interaction_df['ligand'] = lr_interaction_df[
        'genesymbol_intercell_source'].map(adata.var['ensembl_id'])
    lr_interaction_df['receptor'] = lr_interaction_df[
        'genesymbol_intercell_target'].map(adata.var['ensembl_id'])
    lr_interaction_df.drop(
        ['genesymbol_intercell_source', 'genesymbol_intercell_target'],
        axis=1,
        inplace=True)

    # Drop interactions where ligand or receptor is not in the AnnData object
    lr_interaction_df = lr_interaction_df.dropna()

    return lr_interaction_df


# Adapted from /terra/src/app/infer.py (with scGPT metadata information instead of pyensembl
def harmonize_adata(adata: ad.AnnData,
                    ensembl_release: int=110, # 111
                    min_genes_per_cell: int=10,
                    ) -> ad.AnnData:
    """
    Harmonize an AnnData object prior to tokenization.

    Parameters
    -----------
    adata:
        An unharmonized AnnData object.
    ensembl_release:
        Ensembl release used to retrieve ensembl IDs.
    min_genes_per_cell:
        Minimum amount of genes per cell for a cell not to be filtered.

    Returns:
    -----------
    adata:
        A harmonized AnnData object.
    """
    """
    print('==================================================')
    print('STEP 1: ADDING ENSEMBL IDS...')
    print('==================================================')
    print(f'Adding ensembl IDs from release {ensembl_release}...')
    # Extract ensembl IDs of protein coding and miRNA mouse genes
    ensembl = EnsemblRelease(release=ensembl_release, species='human')
    ensembl.download()
    ensembl.index()
    all_genes = ensembl.genes()
    protein_coding_genes = [
        gene for gene in all_genes if gene.biotype == "protein_coding"]
    mirna_genes = [
        gene for gene in all_genes if gene.biotype == "miRNA"]
    all_relevant_genes = protein_coding_genes + mirna_genes
    gene_ensembl_map_dict = {
        gene.gene_name: gene.gene_id for gene in all_relevant_genes}
    """
    metadata_df = pd.read_csv(scgpt_metadata_file)
    gene_ensembl_map_dict = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))

    adata_gene_names = [gene_name for gene_name in adata.var_names.tolist()]
    adata.var.index = adata_gene_names
    
    harmonized_gene_names = []
    matching_ensembl_ids = []
    for gene_name in adata_gene_names:
        if gene_name in gene_ensembl_map_dict.keys():
            harmonized_gene_names.append(gene_name)
            matching_ensembl_ids.append(gene_ensembl_map_dict[gene_name])
    print(f'Number of genes with matching ensembl IDs: \
          {len(harmonized_gene_names)}.')
    print(f'Number of genes skipped: \
          {len(adata_gene_names) - len(harmonized_gene_names)}.')

    adata = adata[:, adata.var.index.isin(harmonized_gene_names)].copy()
    adata.var = pd.DataFrame(
        index=pd.Index(harmonized_gene_names, name="gene_name"),
        data={"ensembl_id": matching_ensembl_ids})

    # Add dummy values as special values
    print('==================================================')
    print('STEP 2: ADDING SPECIAL VALUES...')
    print('==================================================')
    if 'cell_id' not in adata.obs.keys():
        adata.obs['cell_id'] = adata.obs_names
    if 'dataset_id' not in adata.uns.keys():
        adata.uns['dataset_id'] = 14  # just dummy values
    if 'batch' not in adata.uns.keys():
        adata.uns['batch'] = 'batch0' # just dummy values
    if 'assay' not in adata.uns.keys():
        adata.uns['assay'] = 'xenium' # just dummy values
    if 'species' not in adata.uns.keys():
        adata.uns['species'] = 'homo_sapiens' # just dummy values
    if 'tissue' not in adata.uns.keys():
        adata.uns['tissue'] = 'lung' # just dummy values

    # Filter cells with less than min_genes_per_cell genes
    sc.pp.filter_cells(
        adata,
        min_genes=min_genes_per_cell)

    return adata

# 2. Experiments

## 2.1 Retrieve Gene Embeddings

In [5]:
method = 'scGPT'

In [6]:
### Get scGPT gene embeddings for each gene Ensembl id ###
# Not compatible with NVIDIA H100 80GB HBM3 with CUDA capability sm_90
    
# Set config
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
pad_value = -2
n_input_bins = 51

# Get vocab
scgpt_vocab = GeneVocab.from_file(scgpt_vocab_file)
for s in special_tokens:
    if s not in scgpt_vocab:
        scgpt_vocab.append_token(s)

# Retrieve model parameters from config files
with open(scgpt_model_config_file, "r") as f:
    model_configs = json.load(f)
print(f"Resume model from {scgpt_model_file}, the model args will override the "
      f"config {scgpt_model_config_file}.")
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
n_layers_cls = model_configs["n_layers_cls"]

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(scgpt_vocab)  # size of vocabulary
model = TransformerModel(ntokens,
                         embsize,
                         nhead,
                         d_hid,
                         nlayers,
                         vocab=scgpt_vocab,
                         pad_value=pad_value,
                         n_input_bins=n_input_bins,)

try:
    model.load_state_dict(torch.load(scgpt_model_file))
    print(f"Loading all model params from {scgpt_model_file}")
except:
    model_dict = model.state_dict()
    pretrained_dict = torch.load(scgpt_model_file)
    pretrained_dict = {k: v
                       for k, v in pretrained_dict.items()
                       if k in model_dict and v.shape == model_dict[k].shape}
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)
model.to(device)

# Create mapping from Ensembl id to scGPT token id
gene2idx = scgpt_vocab.get_stoi()

metadata_df = pd.read_csv(scgpt_metadata_file)

ensemblid_mapping = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))
ensemblid2idx = {ensemblid_mapping[k]: v for k, v in gene2idx.items() if k in ensemblid_mapping}

# Retrieve the data-independent gene embeddings from scGPT
gene_ids = np.array([id for id in ensemblid2idx.values()])
gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
gene_embeddings = gene_embeddings.detach().cpu().numpy()

# Store gene embeddings with Ensembl id
emsemblid_embedding_mapping = {}
for i, (key, val) in enumerate(ensemblid2idx.items()):
    emsemblid_embedding_mapping[key] = gene_embeddings[i, :]

Resume model from scGPT/checkpoints/best_model.pt, the model args will override the config scGPT/checkpoints/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.bias with shape torch.Size([512])
Loading params transformer_encoder.

## 2.2 xhs1000-39b_1p (Xenium Human Skin Eczema)

In [7]:
# Dataset variables
dataset_name = 'xhs1000-39b_1p'
cell_type_key = 'annotation_new7'
niche_key = 'NICHE_NAMES'
sample_key = 'sample'
batch_key = 'batch'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch11.h5ad', # held out from pretraining
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch19.h5ad', # held out from pretraining
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch32.h5ad', # held out from pretraining
]
spot_size = 20

In [8]:
adata_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata.obs[batch_key] = adata.uns[batch_key]
    adata_list.append(adata)

Number of genes with matching ensembl IDs:           4947.
Number of genes skipped:           1.
STEP 2: ADDING SPECIAL VALUES...


### 2.2.1 Ligand-Receptor Prediction

#### 2.2.1.1 CellPhoneDB

In [9]:
lr_database = 'cellphonedb'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

Downloaded cellphonedb.zip into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded complex_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded gene_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded interaction_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded protein_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded uniprot_synonyms.tsv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources
Downloaded transcription_factor_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources

               ligand         receptor
2     ENSG00000039068  ENSG00000139187
4     ENSG00000170558  ENSG00000104921
232   ENSG00000090339  ENSG00000005844
237   ENSG00000108622  ENSG00000090659
239   ENSG00000076662  ENSG00000090659
...          

In [10]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [11]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(207, 1024)
(207, 1024)


In [12]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 14.9436, Val Loss = 0.7357
Epoch 2: Train Loss = 14.5237, Val Loss = 0.7366
Epoch 3: Train Loss = 14.0359, Val Loss = 0.7401
Epoch 4: Train Loss = 13.6191, Val Loss = 0.7414
Epoch 5: Train Loss = 13.2640, Val Loss = 0.7441
Epoch 6: Train Loss = 12.8989, Val Loss = 0.7490
Epoch 7: Train Loss = 12.6399, Val Loss = 0.7510
Epoch 8: Train Loss = 12.3521, Val Loss = 0.7520
Epoch 9: Train Loss = 12.0702, Val Loss = 0.7562
Epoch 10: Train Loss = 11.7861, Val Loss = 0.7605
Epoch 11: Train Loss = 11.5563, Val Loss = 0.7639
Early stopping at epoch 11

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.5714    0.5217    0.5455        23
           1     0.4762    0.5263    0.5000        19

    accuracy                         0.5238        42
   macro avg     0.5238    0.5240    0.5227        42
weighted avg     0.5283    0.5238    0.5249        42


--- Evaluation Report saved. ---
Using device:

#### 2.2.1.2 OmniPath

In [13]:
lr_database = 'omnipath'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

                ligand         receptor
66     ENSG00000187266  ENSG00000138741
84     ENSG00000010810  ENSG00000133107
111    ENSG00000010810  ENSG00000137672
183    ENSG00000010810  ENSG00000111199
337    ENSG00000133116  ENSG00000165125
...                ...              ...
88693  ENSG00000113520  ENSG00000166888
89474  ENSG00000067182  ENSG00000065559
90565  ENSG00000019991  ENSG00000100030
90821  ENSG00000125538  ENSG00000105397
93513  ENSG00000136244  ENSG00000168610

[1159 rows x 2 columns]


In [14]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [15]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(579, 1024)
(579, 1024)


In [16]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 42.0962, Val Loss = 0.7688
Epoch 2: Train Loss = 40.6114, Val Loss = 0.7629
Epoch 3: Train Loss = 39.4623, Val Loss = 0.7595
Epoch 4: Train Loss = 38.5162, Val Loss = 0.7557
Epoch 5: Train Loss = 37.6679, Val Loss = 0.7570
Epoch 6: Train Loss = 37.0163, Val Loss = 0.7534
Epoch 7: Train Loss = 36.3518, Val Loss = 0.7542
Epoch 8: Train Loss = 35.8031, Val Loss = 0.7536
Epoch 9: Train Loss = 35.2887, Val Loss = 0.7557
Epoch 10: Train Loss = 34.8626, Val Loss = 0.7570
Epoch 11: Train Loss = 34.4422, Val Loss = 0.7560
Epoch 12: Train Loss = 34.0535, Val Loss = 0.7572
Epoch 13: Train Loss = 33.6879, Val Loss = 0.7589
Epoch 14: Train Loss = 33.3877, Val Loss = 0.7603
Epoch 15: Train Loss = 33.0717, Val Loss = 0.7604
Epoch 16: Train Loss = 32.7728, Val Loss = 0.7626
Early stopping at epoch 16

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.5882    0.6250    0.6061        64
           1   

## 2.3 chl59-8b_1p (NanoString CosMx Human Lung NSCLC)

In [17]:
# Dataset variables
dataset_name = 'chl59-8b_1p'
cell_type_key = 'cell_type'
niche_key = 'niche'
sample_key = 'batch'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung12+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung13+SMI+Flat+data.tar.h5ad', # unannotated
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep1+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep2+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep3+SMI+Flat+data.tar.h5ad', # unannotated
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung6+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung9_Rep1+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung9_Rep2+SMI+Flat+data.tar.h5ad'
]
spot_size = 100

In [18]:
adata_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata.obs[sample_key] = adata.uns[sample_key]
    adata_list.append(adata)

Number of genes with matching ensembl IDs:           944.
Number of genes skipped:           2.
STEP 2: ADDING SPECIAL VALUES...


### 2.3.1 Ligand-Receptor Prediction

#### 2.3.1.1 CellPhoneDB

In [19]:
lr_database = 'cellphonedb'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

Downloaded cellphonedb.zip into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded complex_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded gene_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded interaction_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded protein_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded uniprot_synonyms.tsv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources
Downloaded transcription_factor_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources

               ligand         receptor
232   ENSG00000090339  ENSG00000005844
237   ENSG00000108622  ENSG00000090659
239   ENSG00000076662  ENSG00000090659
355   ENSG00000142192  ENSG00000019582
359   ENSG00000142192  ENSG00000146072
...          

In [20]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [21]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(87, 1024)
(87, 1024)


In [22]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 6.4209, Val Loss = 0.7181
Epoch 2: Train Loss = 6.3190, Val Loss = 0.7335
Epoch 3: Train Loss = 6.1473, Val Loss = 0.7450
Epoch 4: Train Loss = 5.9843, Val Loss = 0.7611
Epoch 5: Train Loss = 5.8204, Val Loss = 0.7820
Epoch 6: Train Loss = 5.6043, Val Loss = 0.7976
Epoch 7: Train Loss = 5.5111, Val Loss = 0.8145
Epoch 8: Train Loss = 5.3288, Val Loss = 0.8257
Epoch 9: Train Loss = 5.2022, Val Loss = 0.8427
Epoch 10: Train Loss = 5.0897, Val Loss = 0.8635
Epoch 11: Train Loss = 4.9664, Val Loss = 0.8696
Early stopping at epoch 11

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.2500    0.0909    0.1333        11
           1     0.2857    0.5714    0.3810         7

    accuracy                         0.2778        18
   macro avg     0.2679    0.3312    0.2571        18
weighted avg     0.2639    0.2778    0.2296        18


--- Evaluation Report saved. ---
Using device: cuda
Epoch

#### 2.3.1.2 OmniPath

In [23]:
lr_database = 'omnipath'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

                ligand         receptor
433    ENSG00000187266  ENSG00000096968
473    ENSG00000198719  ENSG00000148400
497    ENSG00000017427  ENSG00000140443
500    ENSG00000108691  ENSG00000157227
607    ENSG00000101384  ENSG00000148400
...                ...              ...
83763  ENSG00000049130  ENSG00000168610
84137  ENSG00000169242  ENSG00000169398
84891  ENSG00000125538  ENSG00000081985
90821  ENSG00000125538  ENSG00000105397
93513  ENSG00000136244  ENSG00000168610

[643 rows x 2 columns]


In [24]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [25]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(321, 1024)
(321, 1024)


In [26]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 22.3997, Val Loss = 0.7346
Epoch 2: Train Loss = 21.7528, Val Loss = 0.7283
Epoch 3: Train Loss = 21.1998, Val Loss = 0.7223
Epoch 4: Train Loss = 20.6550, Val Loss = 0.7265
Epoch 5: Train Loss = 20.2002, Val Loss = 0.7206
Epoch 6: Train Loss = 19.8119, Val Loss = 0.7248
Epoch 7: Train Loss = 19.4354, Val Loss = 0.7268
Epoch 8: Train Loss = 19.1035, Val Loss = 0.7325
Epoch 9: Train Loss = 18.7833, Val Loss = 0.7235
Epoch 10: Train Loss = 18.5026, Val Loss = 0.7350
Epoch 11: Train Loss = 18.2527, Val Loss = 0.7338
Epoch 12: Train Loss = 17.9922, Val Loss = 0.7367
Epoch 13: Train Loss = 17.7785, Val Loss = 0.7422
Epoch 14: Train Loss = 17.5536, Val Loss = 0.7372
Epoch 15: Train Loss = 17.3734, Val Loss = 0.7492
Early stopping at epoch 15

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.4828    0.3889    0.4308        36
           1     0.3889    0.4828    0.4308        29

    accura

## 2.4 xhk1007-41b_1p (Xenium Human Kidney RCC)

In [27]:
# Dataset variables
dataset_name = 'xhk1007-CV1-41b_1p'
cell_type_key = 'level_1_cell_type'
sample_key = 'batch'
test_sample = 'batch14'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/CV1-KID-0-FT-3.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/CV1-KID-0-FT-1.h5ad'
]
spot_size = 30

In [28]:
adata_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata.obs[batch_key] = adata.uns[batch_key]
    adata_list.append(adata)

Number of genes with matching ensembl IDs:           4948.
Number of genes skipped:           1.
STEP 2: ADDING SPECIAL VALUES...


### 2.4.1 Ligand-Receptor Prediction

#### 2.4.1.1 CellPhoneDB

In [29]:
lr_database = 'cellphonedb'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

Downloaded cellphonedb.zip into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded complex_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded gene_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded interaction_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded protein_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded uniprot_synonyms.tsv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources
Downloaded transcription_factor_input.csv into /nfs/team361/sb75/terra-reproducibility/artifacts/cellphonedb/v5.0.0/sources

               ligand         receptor
2     ENSG00000039068  ENSG00000139187
4     ENSG00000170558  ENSG00000104921
232   ENSG00000090339  ENSG00000005844
237   ENSG00000108622  ENSG00000090659
239   ENSG00000076662  ENSG00000090659
...          

In [30]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [31]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(207, 1024)
(207, 1024)


In [32]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 15.0600, Val Loss = 0.6802
Epoch 2: Train Loss = 14.4956, Val Loss = 0.6817
Epoch 3: Train Loss = 14.0720, Val Loss = 0.6851
Epoch 4: Train Loss = 13.6857, Val Loss = 0.6892
Epoch 5: Train Loss = 13.2984, Val Loss = 0.6942
Epoch 6: Train Loss = 12.9767, Val Loss = 0.6995
Epoch 7: Train Loss = 12.6786, Val Loss = 0.7023
Epoch 8: Train Loss = 12.3724, Val Loss = 0.7071
Epoch 9: Train Loss = 12.0846, Val Loss = 0.7109
Epoch 10: Train Loss = 11.8511, Val Loss = 0.7140
Epoch 11: Train Loss = 11.6212, Val Loss = 0.7150
Early stopping at epoch 11

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.5625    0.3913    0.4615        23
           1     0.4615    0.6316    0.5333        19

    accuracy                         0.5000        42
   macro avg     0.5120    0.5114    0.4974        42
weighted avg     0.5168    0.5000    0.4940        42


--- Evaluation Report saved. ---
Using device:

#### 2.4.1.2 OmniPath

In [33]:
lr_database = 'omnipath'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

                ligand         receptor
66     ENSG00000187266  ENSG00000138741
84     ENSG00000010810  ENSG00000133107
111    ENSG00000010810  ENSG00000137672
183    ENSG00000010810  ENSG00000111199
337    ENSG00000133116  ENSG00000165125
...                ...              ...
88693  ENSG00000113520  ENSG00000166888
89474  ENSG00000067182  ENSG00000065559
90565  ENSG00000019991  ENSG00000100030
90821  ENSG00000125538  ENSG00000105397
93513  ENSG00000136244  ENSG00000168610

[1159 rows x 2 columns]


In [34]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [35]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(579, 1024)
(579, 1024)


In [36]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 41.6982, Val Loss = 0.7273
Epoch 2: Train Loss = 40.4195, Val Loss = 0.7269
Epoch 3: Train Loss = 39.3815, Val Loss = 0.7319
Epoch 4: Train Loss = 38.5038, Val Loss = 0.7378
Epoch 5: Train Loss = 37.7348, Val Loss = 0.7375
Epoch 6: Train Loss = 37.1588, Val Loss = 0.7423
Epoch 7: Train Loss = 36.4968, Val Loss = 0.7452
Epoch 8: Train Loss = 36.0038, Val Loss = 0.7480
Epoch 9: Train Loss = 35.4534, Val Loss = 0.7523
Epoch 10: Train Loss = 35.0347, Val Loss = 0.7548
Epoch 11: Train Loss = 34.5592, Val Loss = 0.7588
Epoch 12: Train Loss = 34.1869, Val Loss = 0.7612
Early stopping at epoch 12

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.6500    0.6094    0.6290        64
           1     0.5536    0.5962    0.5741        52

    accuracy                         0.6034       116
   macro avg     0.6018    0.6028    0.6016       116
weighted avg     0.6068    0.6034    0.6044       11